# Mínimos cuadrados matriciales y descenso de gradiente estocástico

Trabajamos con $n = 26$ observaciones $(y_i, x_{i1}, x_{i2})$ y el modelo lineal sin término independiente

$$
\mathbf{y} = X\boldsymbol{\beta} + \boldsymbol{\varepsilon},
\qquad
X = \begin{bmatrix} \mathbf{x}_1 & \mathbf{x}_2 \end{bmatrix} \in \mathbb{R}^{26 \times 2},
\qquad
\boldsymbol{\beta} = \begin{pmatrix} \beta_1 \\ \beta_2 \end{pmatrix},
\qquad
\boldsymbol{\varepsilon} \in \mathbb{R}^{26}.
$$

El recorrido tiene dos etapas:

1. Ajuste por mínimos cuadrados en forma matricial: minimizamos $S(\boldsymbol{\beta}) = \|\mathbf{y} - X\boldsymbol{\beta}\|_2^2$ y llegamos a las ecuaciones normales $X^\top X \hat{\boldsymbol{\beta}} = X^\top \mathbf{y}$.
2. Búsqueda del mismo $\hat{\boldsymbol{\beta}}$ con descenso de gradiente estocástico (SGD), con una animación de la convergencia.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

datos = np.array([
    [-2, 4, 5],
    [-15, 5, -5],
    [0, 0, 4],
    [9, -3, -5],
    [15, -5, 0],
    [-12, 4, -3],
    [0, 0, -2],
    [0, 0, -4],
    [6, -2, -4],
    [-6, 2, 1],
    [3, -1, -4],
    [-12, 4, 1],
    [-12, 4, 2],
    [6, -2, 3],
    [6, -2, -4],
    [-6, 2, 5],
    [-1, 1, 1],
    [-8, 4, 2],
    [-13, 1, -5],
    [4, 0, 2],
    [8, -2, 1],
    [3, -3, -3],
    [5, 1, 4],
    [4, -2, -1],
    [-4, 0, -2],
    [-15, 3, -3],
], dtype=float)

y = datos[:, 0]
X = datos[:, 1:]
n, p = X.shape

print(f"X.shape = {X.shape}")
print(f"y.shape = {y.shape}")

X.shape = (26, 2)
y.shape = (26,)


## 1. Ajuste por mínimos cuadrados (versión matricial)

La función que minimizamos es la suma de cuadrados de los residuos

$$
S(\boldsymbol{\beta}) \;=\; \|\mathbf{y} - X\boldsymbol{\beta}\|_2^2 \;=\; (\mathbf{y} - X\boldsymbol{\beta})^\top (\mathbf{y} - X\boldsymbol{\beta}).
$$

Al expandir el producto, y usando que el escalar $\boldsymbol{\beta}^\top X^\top \mathbf{y}$ coincide con su traspuesta $\mathbf{y}^\top X \boldsymbol{\beta}$, los dos términos medios se suman:

$$
S(\boldsymbol{\beta})
= \mathbf{y}^\top \mathbf{y} - 2\,\boldsymbol{\beta}^\top X^\top \mathbf{y} + \boldsymbol{\beta}^\top X^\top X \boldsymbol{\beta}.
$$

Con las derivadas vectoriales $\nabla_{\boldsymbol{\beta}}(\boldsymbol{\beta}^\top \mathbf{a}) = \mathbf{a}$ y $\nabla_{\boldsymbol{\beta}}(\boldsymbol{\beta}^\top A \boldsymbol{\beta}) = 2A\boldsymbol{\beta}$ para $A$ simétrica, el gradiente es

$$
\nabla_{\boldsymbol{\beta}} S(\boldsymbol{\beta}) = -2\,X^\top \mathbf{y} + 2\,X^\top X \boldsymbol{\beta}.
$$

En el mínimo el gradiente se anula y aparecen las **ecuaciones normales**:

$$
\nabla_{\boldsymbol{\beta}} S(\boldsymbol{\beta}) = \mathbf{0}
\quad\Longleftrightarrow\quad
X^\top X\,\hat{\boldsymbol{\beta}} = X^\top \mathbf{y}.
$$

La hessiana es $2\,X^\top X$, semidefinida positiva. Si $X$ tiene columnas linealmente independientes, $X^\top X$ es invertible y el mínimo es único:

$$
\hat{\boldsymbol{\beta}} = (X^\top X)^{-1} X^\top \mathbf{y}.
$$

En nuestro caso $X^\top X \in \mathbb{R}^{2 \times 2}$ y $X^\top \mathbf{y} \in \mathbb{R}^{2 \times 1}$: un sistema de dos ecuaciones con dos incógnitas.

In [2]:
XtX = X.T @ X
Xty = X.T @ y

beta_hat = np.linalg.solve(XtX, Xty)
beta_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)

print("X.T @ X =")
print(XtX)
print("X.T @ y =", Xty)
print()
print("beta_hat (solve) =", beta_hat)
print("beta_hat (lstsq) =", beta_lstsq)
print("||XtX @ beta_hat - Xty|| =", np.linalg.norm(XtX @ beta_hat - Xty))

X.T @ X =
[[189.  44.]
 [ 44. 282.]]
X.T @ y = [-511.   66.]

beta_hat (solve) = [-2.8621549   0.68061991]
beta_hat (lstsq) = [-2.8621549   0.68061991]
||XtX @ beta_hat - Xty|| = 2.842170943040401e-14


### Calidad del ajuste

Los residuos son $\hat{\boldsymbol{\varepsilon}} = \mathbf{y} - X\hat{\boldsymbol{\beta}}$, la varianza residual estimada vale $\hat{\sigma}^2 = \|\hat{\boldsymbol{\varepsilon}}\|_2^2 / (n - p)$ y el coeficiente de determinación es

$$
R^2 = 1 - \frac{\|\hat{\boldsymbol{\varepsilon}}\|_2^2}{\|\mathbf{y} - \bar{y}\,\mathbf{1}\|_2^2}.
$$

In [3]:
residuos = y - X @ beta_hat
sigma2 = residuos @ residuos / (n - p)
r2 = 1 - (residuos @ residuos) / ((y - y.mean()) @ (y - y.mean()))

print(f"sigma^2 = {sigma2:.4f}")
print(f"R^2     = {r2:.4f}")

sigma^2 = 10.5632
R^2     = 0.8516


## 2. Descenso de gradiente estocástico

Las ecuaciones normales usan el dataset completo a través de $X^\top X$ y $X^\top \mathbf{y}$. El SGD avanza a pasos pequeños y en cada paso observa una sola fila del dataset.

La contribución de la observación $i$ a la suma de cuadrados es

$$
s_i(\boldsymbol{\beta}) = \bigl(y_i - \mathbf{x}_i^\top \boldsymbol{\beta}\bigr)^2,
\qquad
\nabla s_i(\boldsymbol{\beta}) = -2\,\mathbf{x}_i\,\bigl(y_i - \mathbf{x}_i^\top \boldsymbol{\beta}\bigr),
$$

donde $\mathbf{x}_i^\top$ es la fila $i$ de $X$. En el paso $t$ se sortea un índice $i_t$ uniformemente y se da un paso cuesta abajo por el gradiente de esa observación:

$$
\boldsymbol{\beta}_{t+1}
= \boldsymbol{\beta}_t - \eta\,\nabla s_{i_t}(\boldsymbol{\beta}_t)
= \boldsymbol{\beta}_t + 2\eta\,\mathbf{x}_{i_t}\,\bigl(y_{i_t} - \mathbf{x}_{i_t}^\top \boldsymbol{\beta}_t\bigr),
$$

con tamaño de paso $\eta > 0$. Como $\mathbb{E}\bigl[\nabla s_{i_t}(\boldsymbol{\beta})\bigr] = \frac{1}{n}\,\nabla S(\boldsymbol{\beta})$, el paso promedio apunta en dirección de descenso de $S$; la varianza del sorteo hace que la trayectoria zigzaguee y, pasos después, se aglomere alrededor del mínimo.

In [4]:
eta = 0.0025
n_iter = 300
beta = np.array([3.0, -3.0])
rng = np.random.default_rng(42)

trayectoria = [beta.copy()]
perdidas = [float((y - X @ beta) @ (y - X @ beta))]

for _ in range(n_iter):
    i = rng.integers(n)
    xi = X[i]
    beta = beta + 2 * eta * xi * (y[i] - xi @ beta)
    trayectoria.append(beta.copy())
    perdidas.append(float((y - X @ beta) @ (y - X @ beta)))

trayectoria = np.array(trayectoria)
perdidas = np.array(perdidas)

print("beta final (SGD) =", beta)
print("beta (OLS)       =", beta_hat)
print("distancia        =", np.linalg.norm(beta - beta_hat))

beta final (SGD) = [-2.88008024  0.60277819]
beta (OLS)       = [-2.8621549   0.68061991]
distancia        = 0.07987898494271232


### Animación de la convergencia

Cada cuadro corresponde a un paso del SGD. A la izquierda se muestran las curvas de nivel de $S(\boldsymbol{\beta}) = \|\mathbf{y} - X\boldsymbol{\beta}\|_2^2$ (en escala logarítmica) junto con la trayectoria $\boldsymbol{\beta}_0, \boldsymbol{\beta}_1, \boldsymbol{\beta}_2, \dots$; la estrella marca $\hat{\boldsymbol{\beta}}$ obtenido por ecuaciones normales. A la derecha aparece la pérdida $S(\boldsymbol{\beta}_t)$ por iteración, en escala logarítmica.

In [5]:
b1 = np.linspace(beta_hat[0] - 7, beta_hat[0] + 7, 220)
b2 = np.linspace(beta_hat[1] - 7, beta_hat[1] + 7, 220)
B1, B2 = np.meshgrid(b1, b2)
G = np.stack([B1.ravel(), B2.ravel()], axis=1)
S_grid = ((y[None, :] - G @ X.T) ** 2).sum(axis=1).reshape(B1.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6))

cs = ax1.contour(B1, B2, np.log10(S_grid), levels=18, cmap="viridis")
ax1.clabel(cs, inline=True, fontsize=7, fmt="%.1f")
ax1.plot(beta_hat[0], beta_hat[1], "*", color="crimson", markersize=15, zorder=5)
linea, = ax1.plot([], [], "-", color="black", lw=1.3)
punto, = ax1.plot([], [], "o", color="black", markersize=7, zorder=6)
ax1.set_xlim(b1[0], b1[-1])
ax1.set_ylim(b2[0], b2[-1])
ax1.set_xlabel(r"$\beta_1$")
ax1.set_ylabel(r"$\beta_2$")
ax1.set_title("Trayectoria del SGD sobre las curvas de nivel de $S$")

linea_perdida, = ax2.plot([], [], color="steelblue", lw=2)
punto_perdida, = ax2.plot([], [], "o", color="steelblue", markersize=7)
ax2.set_xlim(0, n_iter)
ax2.set_ylim(perdidas.min() * 0.8, perdidas.max() * 1.2)
ax2.set_yscale("log")
ax2.set_xlabel("iteración $t$")
ax2.set_ylabel(r"$S(\beta_t)$")
ax2.set_title("Pérdida por iteración")


def actualizar(t):
    linea.set_data(trayectoria[: t + 1, 0], trayectoria[: t + 1, 1])
    punto.set_data([trayectoria[t, 0]], [trayectoria[t, 1]])
    linea_perdida.set_data(np.arange(t + 1), perdidas[: t + 1])
    punto_perdida.set_data([t], [perdidas[t]])
    return linea, punto, linea_perdida, punto_perdida


anim = FuncAnimation(
    fig,
    actualizar,
    frames=range(0, len(trayectoria), 5),
    interval=60,
    blit=False,
)
plt.close(fig)
plt.rcParams["savefig.dpi"] = 72
HTML(anim.to_jshtml())

In [6]:
print(f"{'':>14}{'beta_1':>12}{'beta_2':>12}")
print(f"{'OLS':>14}{beta_hat[0]:>12.4f}{beta_hat[1]:>12.4f}")
print(f"{'SGD final':>14}{beta[0]:>12.4f}{beta[1]:>12.4f}")
print(f"{'distancia':>14}{np.linalg.norm(beta - beta_hat):>12.4f}")

                    beta_1      beta_2
           OLS     -2.8622      0.6806
     SGD final     -2.8801      0.6028
     distancia      0.0799


## Lo que queda de la sesión

Las ecuaciones normales $X^\top X \hat{\boldsymbol{\beta}} = X^\top \mathbf{y}$ entregan la solución exacta del problema cuadrático de dos incógnitas con una fórmula. El SGD da un paso por observación sorteada y la trayectoria se acerca al mismo punto, con un costo por paso que solo depende de $p$. Esa economía por paso sostiene el entrenamiento de modelos con millones de parámetros, donde formar $X^\top X$ resulta inviable.

El tamaño de paso $\eta$ regula el compromiso entre velocidad de avance y oscilación: con $\eta$ grande el mínimo se alcanza antes y la nube final de iterados alrededor de $\hat{\boldsymbol{\beta}}$ queda más ancha. Volver a ejecutar la celda del SGD con $\eta = 0.01$ deja ese efecto a la vista.